In [22]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
import sys
sys.path.append("..")   # make the project root visible, so src/ can be imported

import pandas as pd
from src.data_cleaning import fix_days_employed

df = pd.read_csv("../data/raw/application_train.csv")
df_clean = fix_days_employed(df)

print(df_clean["DAYS_EMPLOYED"].max())
print(df_clean["DAYS_EMPLOYED_ANOM"].sum())

0.0
55374


In [24]:
print((df_clean["DAYS_EMPLOYED"] == 0).sum())

2


In [25]:
from src.data_cleaning import fix_days_employed, drop_excluded_columns

df_clean = drop_excluded_columns(fix_days_employed(df))
print(df.shape, "→", df_clean.shape)

(307511, 122) → (307511, 73)


In [26]:
building_cols = [c for c in df.columns if c.endswith(("_AVG", "_MODE", "_MEDI"))]
print(len(building_cols))

47


In [27]:
from src.features import add_derived_features

df_clean = add_derived_features(drop_excluded_columns(fix_days_employed(df)))
print(df_clean.shape)
print(df_clean[["CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO",
                "CREDIT_TERM", "CREDIT_GOODS_RATIO", "AGE_YEARS"]].describe())

(307511, 78)
       CREDIT_INCOME_RATIO  ANNUITY_INCOME_RATIO    CREDIT_TERM  \
count        307511.000000         307499.000000  307499.000000   
mean              3.957570              0.180930      21.612322   
std               2.689728              0.094574       7.823823   
min               0.004808              0.000224       8.036674   
25%               2.018667              0.114782      15.614496   
50%               3.265067              0.162833      20.000000   
75%               5.159880              0.229067      27.099985   
max              84.736842              1.875965      45.305079   

       CREDIT_GOODS_RATIO      AGE_YEARS  
count       307233.000000  307511.000000  
mean             1.122995      43.906900  
std              0.124045      11.947950  
min              0.150000      20.503765  
25%              1.000000      33.984942  
50%              1.118800      43.121150  
75%              1.198000      53.886379  
max              6.000000      69.07323

In [28]:
from src.data_split import split_train_test

train, test = split_train_test(df_clean)

print(f"Train: {len(train):,} rows, default rate {train['TARGET'].mean():.2%}")
print(f"Test:  {len(test):,} rows, default rate {test['TARGET'].mean():.2%}")

Train: 215,257 rows, default rate 8.07%
Test:  92,254 rows, default rate 8.07%


In [30]:
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

train.to_parquet(PROCESSED_DIR / "train.parquet", index=False)
test.to_parquet(PROCESSED_DIR / "test.parquet", index=False)

print(pd.read_parquet(PROCESSED_DIR / "train.parquet").shape)

(215257, 78)
